In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
BASE_PATH = "/kaggle/input/two-stage-semiconductor-defect-dataset-sem-img/semiconductor_defect_dataset"

STAGE1_PATH = BASE_PATH + "/stage1_binary_dataset"
STAGE2_PATH = BASE_PATH + "/stage2_multiclass_dataset"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_INITIAL = 15
EPOCHS_FINE = 10


In [ ]:
train_gen1 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen1 = ImageDataGenerator(rescale=1./255)

train_stage1 = train_gen1.flow_from_directory(
    STAGE1_PATH + "/Train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_stage1 = val_gen1.flow_from_directory(
    STAGE1_PATH + "/Validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


In [ ]:
base_model1 = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model1.trainable = False

x = base_model1.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output1 = Dense(1, activation="sigmoid")(x)

stage1_model = Model(inputs=base_model1.input, outputs=output1)

stage1_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# ==============================
# STAGE 1 CONFUSION MATRIX
# ==============================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# IMPORTANT: recreate validation generator with shuffle=False
val_gen1_eval = ImageDataGenerator(rescale=1./255)

val_stage1_eval = val_gen1_eval.flow_from_directory(
    STAGE1_PATH + "/Validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

val_stage1_eval.reset()

# Predictions
y_pred_probs1 = stage1_model.predict(val_stage1_eval)
y_pred1 = (y_pred_probs1 > 0.5).astype(int).flatten()
y_true1 = val_stage1_eval.classes

# Confusion Matrix
cm1 = confusion_matrix(y_true1, y_pred1)

plt.figure(figsize=(6,6))
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Clean", "Defect"],
            yticklabels=["Clean", "Defect"])
plt.title("Stage 1 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Classification Report
print("Stage 1 Classification Report:")
print(classification_report(y_true1, y_pred1, zero_division=0))


In [ ]:
callbacks1 = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("stage1_best_model.keras", save_best_only=True)
]

history_stage1 = stage1_model.fit(
    train_stage1,
    validation_data=val_stage1,
    epochs=EPOCHS_INITIAL,
    callbacks=callbacks1
)


In [ ]:
train_gen2 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen2 = ImageDataGenerator(rescale=1./255)

train_stage2 = train_gen2.flow_from_directory(
    STAGE2_PATH + "/Train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_stage2 = val_gen2.flow_from_directory(
    STAGE2_PATH + "/Validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)


In [ ]:
base_model2 = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model2.trainable = False

x = base_model2.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output2 = Dense(train_stage2.num_classes, activation="softmax")(x)

stage2_model = Model(inputs=base_model2.input, outputs=output2)

stage2_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_stage2.classes),
    y=train_stage2.classes
)

class_weights_dict = dict(enumerate(class_weights))


In [ ]:
# ==============================
# STAGE 2 CONFUSION MATRIX
# ==============================

# IMPORTANT: recreate validation generator with shuffle=False
val_gen2_eval = ImageDataGenerator(rescale=1./255)

val_stage2_eval = val_gen2_eval.flow_from_directory(
    STAGE2_PATH + "/Validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

val_stage2_eval.reset()

# Predictions
predictions2 = stage2_model.predict(val_stage2_eval)
y_pred2 = np.argmax(predictions2, axis=1)
y_true2 = val_stage2_eval.classes

class_names2 = list(val_stage2_eval.class_indices.keys())

# Confusion Matrix
cm2 = confusion_matrix(y_true2, y_pred2)

plt.figure(figsize=(8,8))
sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names2,
            yticklabels=class_names2)
plt.title("Stage 2 Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.show()

# Classification Report
print("Stage 2 Classification Report:")
print(classification_report(
    y_true2,
    y_pred2,
    target_names=class_names2,
    zero_division=0
))


In [ ]:
callbacks2 = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("stage2_best_model.keras", save_best_only=True)
]

history_stage2 = stage2_model.fit(
    train_stage2,
    validation_data=val_stage2,
    epochs=EPOCHS_INITIAL,
    callbacks=callbacks2,
    class_weight=class_weights_dict
)


In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(img_path):

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    stage1_pred = stage1_model.predict(img_array)

    if stage1_pred[0][0] > 0.5:
        print("Defect Detected")
        stage2_pred = stage2_model.predict(img_array)
        class_index = np.argmax(stage2_pred)
        class_label = list(train_stage2.class_indices.keys())[class_index]
        print("Defect Type:", class_label)
    else:
        print("Clean Wafer")
